# 🧠 Week 10 Advanced Reading — Student Version
## LDA vs Sliced Inverse Regression (SIR)

**Context:** The main lecture compared PCA and LDA on a dataset with *discrete* class labels (8 reach directions, healthy/impaired). But what if the response variable is *continuous* — like grip force? LDA requires you to bin the force into categories, losing within-category information. Sliced Inverse Regression (SIR) is a method that finds projection directions for continuous responses without binning.

**Objectives:**
- Understand why discretising a continuous response loses information
- Implement SIR from scratch and see how it relates to LDA
- Compare PCA, LDA, and SIR on the grip force dataset
- Discover that each method wins the task its objective was designed for
- Interpret loading vectors to understand *why* the methods disagree

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pickle
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score, LeaveOneGroupOut

plt.rcParams.update({'figure.figsize': (10, 6), 'font.size': 11,
                      'axes.grid': True, 'grid.alpha': 0.3})
logo = LeaveOneGroupOut()

In [ ]:
from google.colab import files
uploaded = files.upload()  # Upload grip_force_data.pkl

In [ ]:
# Load the grip force dataset
with open('grip_force_data.pkl', 'rb') as f:
    D = pickle.load(f)

rates = D['neural_rates']      # (300, 80) firing rates
forces = D['forces']           # continuous force 5-95% MVC
force_cat = D['force_cat']     # 0=Light, 1=Medium, 2=Strong
subjects = D['subjects']       # 15 subjects, 20 trials each
thresholds = D['thresholds']   # sigmoid thresholds for neurons 0-39
slopes = D['slopes']           # sigmoid slopes

cat_names = ['Light', 'Medium', 'Strong']
cat_colors = ['#2ecc71', '#f39c12', '#e74c3c']

sc = StandardScaler().fit_transform(rates)

print(f'Dataset: {rates.shape[0]} trials, {rates.shape[1]} neurons, '
      f'{len(np.unique(subjects))} subjects')
print(f'Force range: {forces.min():.1f}–{forces.max():.1f}% MVC')
print(f'Categories: Light={sum(force_cat==0)}, Medium={sum(force_cat==1)}, '
      f'Strong={sum(force_cat==2)}')

---

## 🟢 Part 1: The Grip Force Dataset (Advanced Notes §1)

### Exercise 1.1: Visualise the dataset (Advanced Figure A1)

**Learning objective:** Understand the structure of the data — 40 force-tuned neurons with sigmoidal responses (thresholds spanning 20–80% MVC) and 40 nuisance neurons that carry subject-specific baseline variance but no force information.

In [ ]:
# Exercise 1.1: Dataset overview (3-panel figure)
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# TODO: Panel A — plot 3 example sigmoidal tuning curves
#   Use neurons 0, 19, 39 (low/mid/high threshold)
#   First scatter the raw data: ax.scatter(forces, rates[:, idx], ...)
#   Then overlay the smooth sigmoid curve
#   Add vertical dashed lines at 33% and 66% (category boundaries)

# TODO: Panel B — histogram of thresholds across 40 force-tuned neurons
#   Add category boundary lines

# TODO: Panel C — histogram of force values, coloured by category

# YOUR CODE HERE
plt.tight_layout()
plt.show()

### Exercise 1.2: Why binning loses information

**Learning objective:** See that two trials in the same category (e.g. 34% and 65% MVC, both 'Medium') produce very different neural activity. LDA treats them as identical.

In [ ]:
# Exercise 1.2: Within-category variance
# Pick a neuron with threshold near 50% MVC
mid_neuron = np.argmin(np.abs(thresholds - 50))

# TODO: Create 2-panel figure showing:
#   (A) Firing rate vs continuous force — smooth sigmoid visible
#   (B) Same data but x-axis is just the 3 category labels — gradient lost
# Colour points by category in both panels
# YOUR CODE HERE

---

## 🟡 Part 2: Implementing SIR from Scratch (Advanced Notes §2)

### Exercise 2.1: Implement SIR

**Learning objective:** Code SIR step by step and understand how it differs from LDA.

**The SIR algorithm:**
1. Standardise X (zero mean, unit variance)
2. Sort the continuous response y and divide into H evenly-populated slices
3. Compute the slice mean **m̄ₕ** = E[X | y ∈ slice h] for each slice
4. Build the weighted covariance of slice means: M = Σₕ (nₕ/n) (m̄ₕ - x̄)(m̄ₕ - x̄)ᵀ
5. Solve the generalised eigenvalue problem Σ⁻¹M v = λv
6. The top eigenvectors are the SIR directions

**Key insight:** If you set H = K (number of classes) and use categorical labels instead of slices, step 4 becomes the between-class scatter S_B and SIR reduces to LDA.

In [ ]:
# Exercise 2.1: Implement SIR from scratch
def sir(X, y, n_directions=2, n_slices=10):
    """
    Sliced Inverse Regression.
    """
    n, p = X.shape
    
    # Step 1: Estimate total covariance
    Sigma = np.cov(X, rowvar=False)
    
    # Step 2: Create slices by percentiles of y
    percentiles = np.linspace(0, 100, n_slices + 1)
    bins = np.percentile(y, percentiles)
    bins[0] -= 1; bins[-1] += 1
    slice_labels = np.clip(np.digitize(y, bins) - 1, 0, n_slices - 1)
    
    # TODO: Step 3-4: Compute M = weighted covariance of slice means
    # For each slice h: slice_mean = X[mask].mean(0) - grand_mean
    # M += (n_h / n) * outer(slice_mean, slice_mean)
    grand_mean = X.mean(axis=0)
    M = np.zeros((p, p))
    # YOUR CODE HERE
    
    # TODO: Step 5: Solve Σ⁻¹M v = λv
    # Hint: Sigma_inv = np.linalg.inv(Sigma + 1e-6 * np.eye(p))
    # eigvals, eigvecs = np.linalg.eigh(Sigma_inv @ M)
    # YOUR CODE HERE
    
    # Step 6: Return top directions
    idx = np.argsort(eigvals)[::-1]
    return eigvecs[:, idx[:n_directions]], eigvals[idx[:n_directions]]

sir_dirs, sir_evals = sir(sc, forces, n_directions=2, n_slices=10)
print(f'SIR eigenvalues: {sir_evals[0]:.3f}, {sir_evals[1]:.3f}')

### Exercise 2.2: Verify SIR reduces to LDA for categorical responses

**Learning objective:** Show that when you run SIR with 3 slices matching the 3 force categories, the resulting directions are parallel to LDA's directions.

In [ ]:
# Exercise 2.2: SIR with 3 slices ≈ LDA with 3 classes
# TODO: Run SIR with n_slices=3 using force_cat as the response
# TODO: Run LDA with force_cat as labels
# TODO: Compare directions via cosine similarity
# Expected: cosine similarity ≈ 1.0 (parallel directions)
# YOUR CODE HERE

---

## 🟡 Part 3: Three Projections Compared (Advanced Notes §3)

### Exercise 3.1: 2D scatter comparison (Advanced Figure A2)

**Learning objective:** See how PCA, LDA, and SIR project the same data into 2D. Colour by continuous force to see which method best preserves the force gradient.

In [ ]:
# Exercise 3.1: Three projections side by side
# TODO: Compute PCA(2), LDA(2), and SIR(2) projections
# TODO: Scatter plot each, coloured by continuous force
# YOUR CODE HERE

### Exercise 3.2: First axis vs force (Advanced Figure A3)

**Learning objective:** Compare how the first projection axis from each method tracks the continuous force. LDA's LD1 should show a step-like pattern (3 categories), while SIR's first axis should track force smoothly.

In [ ]:
# Exercise 3.2: First projection axis vs continuous force
# TODO: For each method, scatter the first projection score vs force
# Colour by category, fit a regression line, show R² in legend
# YOUR CODE HERE

---

## 🔴 Part 4: Each Method Wins Its Own Game (Advanced Notes §3–4)

### Exercise 4.1: LOSO evaluation (Advanced Figure A4)

**Learning objective:** Evaluate all three methods on two tasks: (1) continuous force regression (R²) and (2) 3-category classification (accuracy). Each method should win the task its objective was designed for.

In [ ]:
# Exercise 4.1: LOSO evaluation on two tasks
# TODO: For each 2D projection (PCA, LDA, SIR):
#   (1) LOSO R² for continuous force regression (LinearRegression)
#   (2) LOSO accuracy for 3-category classification (KNeighborsClassifier(5))
# Plot side-by-side bar charts
# YOUR CODE HERE

### 🤔 Thought Exercise
PCA wins on continuous R², LDA wins on classification. Neither is "best" — it depends on the question you're asking. If a clinician says *"I need to track a patient's grip force continuously for rehabilitation,"* which method would you recommend and why?

---

## 🔴 Part 5: Why Do LDA and SIR Disagree? (Advanced Notes §4)

### Exercise 5.1: Subspace angle (Advanced Figure A5)

**Learning objective:** Compute the angle between LDA's and SIR's first projection directions. A large angle means they found genuinely different structure in the data.

In [ ]:
# Exercise 5.1: Subspace angle between LDA and SIR
from scipy.linalg import subspace_angles

lda_dirs = lda.scalings_[:, :2]

# TODO: Compute principal angles between LDA and SIR subspaces
# Hint: angles = np.rad2deg(subspace_angles(lda_dirs, sir_dirs))

# TODO: Visualise as unit-circle diagram:
#   1. Project 80D directions to 2D via PCA on the loading vectors
#   2. Normalise to unit length
#   3. Draw arrows from origin + an arc showing the angle
# YOUR CODE HERE

### Exercise 5.2: Loading vectors (Advanced Figure A6)

**Learning objective:** Understand *why* LDA and SIR disagree by examining which neurons each method weights. The loading vectors reveal the difference: LDA emphasises neurons near category boundaries (33%, 66% MVC), while SIR distributes weight more evenly across all force-tuned neurons.

In [ ]:
# Exercise 5.2: Compare loading vectors
# TODO: Normalise LD1 and SIR dir1 loading vectors
# TODO: Plot as bar charts, colour red (force-tuned) vs grey (nuisance)
# TODO: Compute loading energy (sum of squared weights) for each group
# YOUR CODE HERE

### Exercise 5.3: Threshold analysis

**Learning objective:** Directly test the claim that LDA weights neurons near category boundaries more heavily.

In [ ]:
# Exercise 5.3: Weight vs distance to category boundary
boundary_dist = np.minimum(np.abs(thresholds - 33), np.abs(thresholds - 66))

# TODO: Scatter |LDA weight| vs boundary distance for force-tuned neurons
# TODO: Scatter |SIR weight| vs boundary distance for force-tuned neurons
# TODO: Compute correlation for each
# Expected: LDA has negative correlation (closer → higher weight)
#           SIR has weaker/no correlation
# YOUR CODE HERE

---

## Summary

In this advanced reading lab you:

1. **Explored the grip force dataset** and saw how binning a continuous response into categories loses within-category information
2. **Implemented SIR from scratch** and verified it reduces to LDA when the response is categorical
3. **Compared PCA, LDA, and SIR** on the same data and found each method wins the task its objective was designed for:
   - PCA → best continuous R² (signal dominates variance)
   - LDA → best category accuracy (directly optimises separation)
   - SIR → between them, competitive on both
4. **Measured the subspace angle** between LDA and SIR (~71°) — they find genuinely different projections of the same data
5. **Interpreted loading vectors** and discovered that LDA emphasises neurons near category boundaries while SIR distributes weight across the full tuning continuum

**Key takeaway:** The right projection method depends on the question. If your response is truly categorical, LDA is optimal. If it's continuous, SIR extracts more information. PCA is always a useful unsupervised baseline.